In [1]:
import os
import time
import json
import random
import tracemalloc
import numpy as np
import pandas as pd
from concurrent.futures import ThreadPoolExecutor

PROJECT_ROOT = r"C:\Users\aseem\OneDrive\Dokumen\aerospatium"
PROCESSED_DATA_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
REPORTS_DIR = os.path.join(PROJECT_ROOT, "reports")
os.makedirs(REPORTS_DIR, exist_ok=True)

# 1. Prepare Benchmark Origin/Destination Pool
origin_ids = [1001, 1002, 1008, 1007, 1014]
destination_ids = [1004, 1006]

# Generate 100 randomized queries
random.seed(42)
queries = [
    (random.choice(origin_ids), random.choice(destination_ids), random.uniform(0.0, 7200.0))
    for _ in range(100)
]

def execute_routing_query(q):
    start_u, goal_v, dep_t = q
    start_time = time.perf_counter()
    p_route, s_route = find_disjoint_corridors(start_u, goal_v, departure_time_s=dep_t)
    duration_ms = (time.perf_counter() - start_time) * 1000.0
    return duration_ms

# 2. Concurrency Load Test across Thread Counts [1, 2, 4, 8]
thread_counts = [1, 2, 4, 8]
benchmark_results = {}

print("--- Running Concurrency Load Benchmarks (100 Queries per Run) ---")
tracemalloc.start()

for tc in thread_counts:
    latencies = []
    t_start = time.perf_counter()
    
    with ThreadPoolExecutor(max_workers=tc) as executor:
        results = list(executor.map(execute_routing_query, queries))
        latencies.extend(results)
        
    total_duration_s = time.perf_counter() - t_start
    throughput = len(queries) / total_duration_s
    
    lat_p50 = float(np.percentile(latencies, 50))
    lat_p95 = float(np.percentile(latencies, 95))
    lat_p99 = float(np.percentile(latencies, 99))
    
    benchmark_results[f"threads_{tc}"] = {
        "thread_count": tc,
        "throughput_qps": throughput,
        "p50_latency_ms": lat_p50,
        "p95_latency_ms": lat_p95,
        "p99_latency_ms": lat_p99,
        "total_time_s": total_duration_s
    }
    
    print(f"Workers: {tc:<2} | Throughput: {throughput:6.1f} QPS | P50: {lat_p50:5.2f} ms | P95: {lat_p95:5.2f} ms | P99: {lat_p99:5.2f} ms")

current_mem_bytes, peak_mem_bytes = tracemalloc.get_traced_memory()
tracemalloc.stop()

peak_mem_mb = peak_mem_bytes / (1024 * 1024)
benchmark_results["memory_profiling"] = {
    "peak_memory_mb": peak_mem_mb,
    "ram_ceiling_gb": 15.0,
    "memory_safety_pass": peak_mem_mb < (15.0 * 1024)
}

print("\n=======================================================")
print("BENCHMARK & RESOURCE PROFILE SUMMARY")
print("=======================================================")
print(f"Peak Memory Allocated: {peak_mem_mb:.2f} MB (Ceiling: 15,360 MB)")
print(f"P95 Latency under 8-thread load: {benchmark_results['threads_8']['p95_latency_ms']:.2f} ms")
print(f"Max Throughput: {benchmark_results['threads_8']['throughput_qps']:.1f} Queries/sec")
print("=======================================================")

# Save JSON benchmark report
report_path = os.path.join(REPORTS_DIR, "benchmark_report.json")
with open(report_path, "w") as f:
    json.dump(benchmark_results, f, indent=2)

print(f"Saved benchmark report to: {report_path}")

--- Running Concurrency Load Benchmarks (100 Queries per Run) ---


NameError: name 'find_disjoint_corridors' is not defined

In [2]:
import os
import math
import heapq
import time
import json
import random
import tracemalloc
import numpy as np
import pandas as pd
from concurrent.futures import ThreadPoolExecutor

PROJECT_ROOT = r"C:\Users\aseem\OneDrive\Dokumen\aerospatium"
PROCESSED_DATA_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
REPORTS_DIR = os.path.join(PROJECT_ROOT, "reports")
os.makedirs(REPORTS_DIR, exist_ok=True)

# 1. Load Road Network Data
df_nodes = pd.read_parquet(os.path.join(PROCESSED_DATA_DIR, "road_nodes.parquet"))
df_edges = pd.read_parquet(os.path.join(PROCESSED_DATA_DIR, "road_edges.parquet"))

node_dict = {row["node_id"]: row.to_dict() for _, row in df_nodes.iterrows()}
edge_dict = {row["edge_id"]: row.to_dict() for _, row in df_edges.iterrows()}

adj_list = {row["node_id"]: [] for _, row in df_nodes.iterrows()}
for _, row in df_edges.iterrows():
    adj_list[row["u"]].append((row["v"], row["edge_id"]))

MAX_NETWORK_SPEED_MPS = 110.0 * (1000.0 / 3600.0)

def haversine_m(lon1, lat1, lon2, lat2):
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlam = math.radians(lon2 - lon1)
    a = math.sin(dphi/2.0)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlam/2.0)**2
    return 6371000.0 * 2.0 * math.atan2(math.sqrt(a), math.sqrt(1.0 - a))

def haversine_heuristic(u, goal):
    u_node = node_dict[u]
    g_node = node_dict[goal]
    dist = haversine_m(u_node["lon"], u_node["lat"], g_node["lon"], g_node["lat"])
    return dist / MAX_NETWORK_SPEED_MPS

def get_edge_weight(edge_id, current_time_s, edge_penalties=None):
    e = edge_dict[edge_id]
    t_hr = current_time_s / 3600.0
    if t_hr <= 1.5:
        base_w = e["weight_t0_s"]
    elif t_hr <= 4.5:
        base_w = e["weight_t3_s"]
    elif t_hr <= 9.0:
        base_w = e["weight_t6_s"]
    else:
        base_w = e["weight_t12_s"]
    penalty = edge_penalties.get(edge_id, 1.0) if edge_penalties else 1.0
    return base_w * penalty

def time_dependent_astar(start_node, goal_node, departure_time_s=0.0, edge_penalties=None):
    pq = []
    h0 = haversine_heuristic(start_node, goal_node)
    heapq.heappush(pq, (h0, 0.0, start_node, [start_node], []))
    visited = {}
    
    while pq:
        f, g, u, path_nodes, path_edges = heapq.heappop(pq)
        if u == goal_node:
            burn_risk = sum(edge_dict[eid]["burn_prob_t3"] for eid in path_edges)
            return path_nodes, path_edges, g, burn_risk
            
        if u in visited and visited[u] <= g:
            continue
        visited[u] = g
        
        for v, eid in adj_list[u]:
            w = get_edge_weight(eid, departure_time_s + g, edge_penalties)
            if w >= 1e8:
                continue
            new_g = g + w
            new_f = new_g + haversine_heuristic(v, goal_node)
            heapq.heappush(pq, (new_f, new_g, v, path_nodes + [v], path_edges + [eid]))
            
    return None, None, float("inf"), float("inf")

def find_disjoint_corridors(start_node, goal_node, departure_time_s=0.0, max_overlap_frac=0.10):
    p_nodes, p_edges, p_time, p_risk = time_dependent_astar(start_node, goal_node, departure_time_s)
    if not p_edges:
        return None, None
        
    penalties = {eid: 50.0 for eid in p_edges}
    s_nodes, s_edges, s_time, s_risk = time_dependent_astar(start_node, goal_node, departure_time_s, edge_penalties=penalties)
    
    if s_edges:
        overlap_edges = set(p_edges).intersection(set(s_edges))
        overlap_frac = len(overlap_edges) / max(1, len(p_edges))
    else:
        s_nodes, s_edges, s_time, s_risk = [], [], float("inf"), float("inf")
        overlap_frac = 0.0
        
    primary_route = {
        "nodes": p_nodes,
        "edges": p_edges,
        "travel_time_s": p_time,
        "burn_risk": p_risk
    }
    secondary_route = {
        "nodes": s_nodes,
        "edges": s_edges,
        "travel_time_s": s_time,
        "burn_risk": s_risk,
        "overlap_fraction": overlap_frac
    }
    return primary_route, secondary_route

# 2. Benchmark Origin/Destination Pool
origin_ids = [1001, 1002, 1008, 1007, 1014]
destination_ids = [1004, 1006]

random.seed(42)
queries = [
    (random.choice(origin_ids), random.choice(destination_ids), random.uniform(0.0, 7200.0))
    for _ in range(100)
]

def execute_routing_query(q):
    start_u, goal_v, dep_t = q
    start_time = time.perf_counter()
    p_route, s_route = find_disjoint_corridors(start_u, goal_v, departure_time_s=dep_t)
    duration_ms = (time.perf_counter() - start_time) * 1000.0
    return duration_ms

# 3. Concurrency Load Test across Thread Counts [1, 2, 4, 8]
thread_counts = [1, 2, 4, 8]
benchmark_results = {}

print("--- Running Concurrency Load Benchmarks (100 Queries per Run) ---")
tracemalloc.start()

for tc in thread_counts:
    latencies = []
    t_start = time.perf_counter()
    
    with ThreadPoolExecutor(max_workers=tc) as executor:
        results = list(executor.map(execute_routing_query, queries))
        latencies.extend(results)
        
    total_duration_s = time.perf_counter() - t_start
    throughput = len(queries) / total_duration_s
    
    lat_p50 = float(np.percentile(latencies, 50))
    lat_p95 = float(np.percentile(latencies, 95))
    lat_p99 = float(np.percentile(latencies, 99))
    
    benchmark_results[f"threads_{tc}"] = {
        "thread_count": tc,
        "throughput_qps": throughput,
        "p50_latency_ms": lat_p50,
        "p95_latency_ms": lat_p95,
        "p99_latency_ms": lat_p99,
        "total_time_s": total_duration_s
    }
    
    print(f"Workers: {tc:<2} | Throughput: {throughput:6.1f} QPS | P50: {lat_p50:5.2f} ms | P95: {lat_p95:5.2f} ms | P99: {lat_p99:5.2f} ms")

current_mem_bytes, peak_mem_bytes = tracemalloc.get_traced_memory()
tracemalloc.stop()

peak_mem_mb = peak_mem_bytes / (1024 * 1024)
benchmark_results["memory_profiling"] = {
    "peak_memory_mb": peak_mem_mb,
    "ram_ceiling_gb": 15.0,
    "memory_safety_pass": peak_mem_mb < (15.0 * 1024)
}

print("\n=======================================================")
print("BENCHMARK & RESOURCE PROFILE SUMMARY")
print("=======================================================")
print(f"Peak Memory Allocated: {peak_mem_mb:.2f} MB (Ceiling: 15,360 MB)")
print(f"P95 Latency under 8-thread load: {benchmark_results['threads_8']['p95_latency_ms']:.2f} ms")
print(f"Max Throughput: {benchmark_results['threads_8']['throughput_qps']:.1f} Queries/sec")
print("=======================================================")

report_path = os.path.join(REPORTS_DIR, "benchmark_report.json")
with open(report_path, "w") as f:
    json.dump(benchmark_results, f, indent=2)

print(f"Saved benchmark report to: {report_path}")

--- Running Concurrency Load Benchmarks (100 Queries per Run) ---
Workers: 1  | Throughput: 2139.8 QPS | P50:  0.41 ms | P95:  0.84 ms | P99:  0.97 ms
Workers: 2  | Throughput: 1614.0 QPS | P50:  0.63 ms | P95:  1.16 ms | P99: 16.39 ms
Workers: 4  | Throughput: 1861.2 QPS | P50:  0.58 ms | P95:  0.96 ms | P99: 13.28 ms
Workers: 8  | Throughput: 1808.8 QPS | P50:  0.58 ms | P95:  0.87 ms | P99:  6.88 ms

BENCHMARK & RESOURCE PROFILE SUMMARY
Peak Memory Allocated: 10.22 MB (Ceiling: 15,360 MB)
P95 Latency under 8-thread load: 0.87 ms
Max Throughput: 1808.8 Queries/sec
Saved benchmark report to: C:\Users\aseem\OneDrive\Dokumen\aerospatium\reports\benchmark_report.json


In [3]:
import os
import json
import numpy as np
import pandas as pd
import geopandas as gpd

PROJECT_ROOT = r"C:\Users\aseem\OneDrive\Dokumen\aerospatium"
PROCESSED_DATA_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
REPORTS_DIR = os.path.join(PROJECT_ROOT, "reports")

# 1. Load All Core Spatial Artifacts
df_static = pd.read_parquet(os.path.join(PROCESSED_DATA_DIR, "h3_static_features.parquet"))
df_surv = pd.read_parquet(os.path.join(PROCESSED_DATA_DIR, "survival_predictions_grid.parquet"))
df_ca = pd.read_parquet(os.path.join(PROCESSED_DATA_DIR, "ca_simulation_camp_fire.parquet"))
df_nodes = pd.read_parquet(os.path.join(PROCESSED_DATA_DIR, "road_nodes.parquet"))
df_edges = pd.read_parquet(os.path.join(PROCESSED_DATA_DIR, "road_edges.parquet"))

with open(os.path.join(PROCESSED_DATA_DIR, "evacuation_routes.json"), "r") as f:
    routes_data = json.load(f)

# Merge grid data
df_grid = df_static.merge(df_surv[["h3_id", "burn_prob_3h", "burn_prob_6h", "burn_prob_12h"]], on="h3_id")
df_grid = df_grid.merge(df_ca[["h3_id", "predicted_arrival_min"]], on="h3_id")

# 2. Build GeoJSON for H3 Risk Contours
h3_features = []
for _, r in df_grid.iterrows():
    # Convert polygon geometry from WKT
    poly_geom = gpd.GeoSeries.from_wkt([r["geometry_wkt"]]).iloc[0]
    coords = [[[lon, lat] for lon, lat in poly_geom.exterior.coords]]
    
    h3_features.append({
        "type": "Feature",
        "geometry": {
            "type": "Polygon",
            "coordinates": coords
        },
        "properties": {
            "h3_id": str(r["h3_id"]),
            "elevation": float(r["elevation_mean_m"]),
            "slope": float(r["slope_mean_deg"]),
            "pop": int(r["population_total"]),
            "prob_0h": 0.0,
            "prob_3h": float(r["burn_prob_3h"]),
            "prob_6h": float(r["burn_prob_6h"]),
            "prob_12h": float(r["burn_prob_12h"]),
            "arrival_min": float(r["predicted_arrival_min"]) if r["predicted_arrival_min"] < 1e8 else -1.0
        }
    })

geojson_h3 = {"type": "FeatureCollection", "features": h3_features}

# 3. Build GeoJSON for Road Network and Closures
road_features = []
for _, r in df_edges.iterrows():
    line_geom = gpd.GeoSeries.from_wkt([r["geometry_wkt"]]).iloc[0]
    coords = [[lon, lat] for lon, lat in line_geom.coords]
    
    road_features.append({
        "type": "Feature",
        "geometry": {
            "type": "LineString",
            "coordinates": coords
        },
        "properties": {
            "edge_id": int(r["edge_id"]),
            "name": str(r["name"]),
            "highway": str(r["highway_type"]),
            "lanes": int(r["lanes"]),
            "speed": float(r["speed_kmh"]),
            "closed_0h": False,
            "closed_3h": bool(r["weight_t3_s"] >= 1e8),
            "closed_6h": bool(r["weight_t6_s"] >= 1e8),
            "closed_12h": bool(r["weight_t12_s"] >= 1e8)
        }
    })

geojson_roads = {"type": "FeatureCollection", "features": road_features}

# 4. Build GeoJSON for Evacuation Routes (Primary and Secondary)
route_features = []
node_coords_map = {int(r["node_id"]): [float(r["lon"]), float(r["lat"])] for _, r in df_nodes.iterrows()}

for idx, plan in enumerate(routes_data):
    # Primary path
    p_nodes = plan["primary_corridor"]["nodes"]
    p_coords = [node_coords_map[nid] for nid in p_nodes]
    route_features.append({
        "type": "Feature",
        "geometry": {"type": "LineString", "coordinates": p_coords},
        "properties": {
            "type": "primary",
            "origin": plan["origin_name"],
            "destination": plan["destination_name"],
            "evacuees": plan["evacuees"],
            "time_min": float(plan["primary_corridor"]["travel_time_s"] / 60.0)
        }
    })
    
    # Secondary path
    s_nodes = plan["secondary_corridor"]["nodes"]
    if s_nodes:
        s_coords = [node_coords_map[nid] for nid in s_nodes]
        route_features.append({
            "type": "Feature",
            "geometry": {"type": "LineString", "coordinates": s_coords},
            "properties": {
                "type": "secondary",
                "origin": plan["origin_name"],
                "destination": plan["destination_name"],
                "evacuees": plan["evacuees"],
                "time_min": float(plan["secondary_corridor"]["travel_time_s"] / 60.0),
                "overlap": float(plan["secondary_corridor"]["overlap_fraction"] * 100.0)
            }
        })

geojson_routes = {"type": "FeatureCollection", "features": route_features}

# 5. Build GeoJSON for Node Markers
node_features = []
for _, r in df_nodes.iterrows():
    is_origin = any(p["origin_name"] == r["name"] for p in routes_data)
    is_dest = any(p["destination_name"] == r["name"] for p in routes_data)
    
    node_type = "origin" if is_origin else ("destination" if is_dest else "junction")
    
    node_features.append({
        "type": "Feature",
        "geometry": {"type": "Point", "coordinates": [float(r["lon"]), float(r["lat"])]},
        "properties": {
            "node_id": int(r["node_id"]),
            "name": str(r["name"]),
            "type": node_type
        }
    })

geojson_nodes = {"type": "FeatureCollection", "features": node_features}

# 6. Generate Standalone MapLibre GL HTML Dashboard
html_template = f"""<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8" />
    <title>AeroSpatium: Wildfire Progression & Evacuation Corridor Dashboard</title>
    <meta name="viewport" content="initial-scale=1,maximum-scale=1,user-scalable=no" />
    <script src="https://unpkg.com/maplibre-gl@3.6.2/dist/maplibre-gl.js"></script>
    <link href="https://unpkg.com/maplibre-gl@3.6.2/dist/maplibre-gl.css" rel="stylesheet" />
    <style>
        body {{ margin: 0; padding: 0; font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif; background: #111; color: #fff; }}
        #map {{ position: absolute; top: 0; bottom: 0; width: 100%; }}
        .panel {{
            position: absolute; top: 20px; left: 20px; z-index: 10;
            background: rgba(18, 24, 38, 0.92); backdrop-filter: blur(8px);
            padding: 20px; border-radius: 12px; border: 1px solid rgba(255,255,255,0.15);
            max-width: 360px; box-shadow: 0 10px 30px rgba(0,0,0,0.5);
        }}
        h2 {{ margin: 0 0 8px 0; font-size: 18px; color: #ff5722; }}
        p {{ margin: 0 0 14px 0; font-size: 12px; color: #aaa; line-height: 1.4; }}
        .slider-container {{ margin-bottom: 16px; }}
        .slider-label {{ font-size: 13px; font-weight: bold; margin-bottom: 6px; display: flex; justify-content: space-between; }}
        input[type=range] {{ width: 100%; accent-color: #ff5722; }}
        .stats-grid {{ display: grid; grid-template-columns: 1fr 1fr; gap: 8px; margin-bottom: 16px; }}
        .stat-card {{ background: rgba(255,255,255,0.05); padding: 8px 12px; border-radius: 6px; }}
        .stat-value {{ font-size: 16px; font-weight: bold; color: #4fc3f7; }}
        .stat-label {{ font-size: 10px; color: #888; text-transform: uppercase; }}
        .legend {{ font-size: 11px; }}
        .legend-item {{ display: flex; align-items: center; margin-bottom: 4px; }}
        .legend-color {{ width: 14px; height: 14px; border-radius: 3px; margin-right: 8px; }}
    </style>
</head>
<body>
<div id="map"></div>
<div class="panel">
    <h2>AeroSpatium Evacuation Engine</h2>
    <p>Real-Time Wildfire Progression, Dynamic BPR Road Closures & Disjoint Evacuation Corridors</p>
    
    <div class="slider-container">
        <div class="slider-label">
            <span>Forecast Horizon:</span>
            <span id="time-display" style="color: #ff5722;">T = 0 Hours (Ignition)</span>
        </div>
        <input type="range" id="time-slider" min="0" max="3" step="1" value="0">
    </div>

    <div class="stats-grid">
        <div class="stat-card">
            <div class="stat-value" id="burned-stat">0 %</div>
            <div class="stat-label">AOI Burned</div>
        </div>
        <div class="stat-card">
            <div class="stat-value" id="closures-stat">0 / 34</div>
            <div class="stat-label">Road Closures</div>
        </div>
        <div class="stat-card">
            <div class="stat-value">3.97 hrs</div>
            <div class="stat-label">Regional T95 Clearance</div>
        </div>
        <div class="stat-card">
            <div class="stat-value">0.87 ms</div>
            <div class="stat-label">Router P95 Latency</div>
        </div>
    </div>

    <div class="legend">
        <div class="legend-item"><div class="legend-color" style="background: #00e676;"></div> Primary Escape Corridor</div>
        <div class="legend-item"><div class="legend-color" style="background: #00b0ff;"></div> Secondary Backup Corridor (≤10% Overlap)</div>
        <div class="legend-item"><div class="legend-color" style="background: #ff1744;"></div> Impassable Road Closure (W >= ∞)</div>
        <div class="legend-item"><div class="legend-color" style="background: rgba(255, 87, 34, 0.7);"></div> High Burn Probability Hexagon</div>
    </div>
</div>

<script>
const h3Data = {json.dumps(geojson_h3)};
const roadsData = {json.dumps(geojson_roads)};
const routesData = {json.dumps(geojson_routes)};
const nodesData = {json.dumps(geojson_nodes)};

const map = new maplibregl.Map({{
    container: 'map',
    style: 'https://basemaps.cartocdn.com/gl/dark-matter-gl-style/style.json',
    center: [-121.611, 39.759],
    zoom: 10.5,
    pitch: 45,
    bearing: -15
}});

map.on('load', () => {{
    // Add H3 Wildfire Hazard Layer
    map.addSource('h3-grid', {{ type: 'geojson', data: h3Data }});
    map.addLayer({{
        'id': 'h3-extrusion',
        'type': 'fill-extrusion',
        'source': 'h3-grid',
        'paint': {{
            'fill-extrusion-color': [
                'interpolate', ['linear'], ['get', 'prob_0h'],
                0.0, 'rgba(0,0,0,0)',
                0.2, '#ffeb3b',
                0.5, '#ff9800',
                0.75, '#f44336',
                1.0, '#b71c1c'
            ],
            'fill-extrusion-height': ['*', ['get', 'prob_0h'], 1800],
            'fill-extrusion-opacity': 0.75
        }}
    }});

    // Add Base Road Network
    map.addSource('roads', {{ type: 'geojson', data: roadsData }});
    map.addLayer({{
        'id': 'roads-base',
        'type': 'line',
        'source': 'roads',
        'paint': {{
            'line-color': '#555',
            'line-width': 2.5,
            'line-opacity': 0.6
        }}
    }});

    // Add Road Closures Layer (Highlighting closed roads)
    map.addLayer({{
        'id': 'roads-closed',
        'type': 'line',
        'source': 'roads',
        'paint': {{
            'line-color': '#ff1744',
            'line-width': 5.0,
            'line-dasharray': [2, 1]
        }},
        'filter': ['==', 'closed_0h', true]
    }});

    // Add Evacuation Routes Layer
    map.addSource('routes', {{ type: 'geojson', data: routesData }});
    map.addLayer({{
        'id': 'routes-layer',
        'type': 'line',
        'source': 'routes',
        'paint': {{
            'line-color': [
                'case',
                ['==', ['get', 'type'], 'primary'], '#00e676',
                '#00b0ff'
            ],
            'line-width': 4.5,
            'line-opacity': 0.95
        }}
    }});

    // Add Evacuation Nodes
    map.addSource('nodes', {{ type: 'geojson', data: nodesData }});
    map.addLayer({{
        'id': 'nodes-circle',
        'type': 'circle',
        'source': 'nodes',
        'paint': {{
            'circle-radius': 7,
            'circle-color': [
                'case',
                ['==', ['get', 'type'], 'origin'], '#ff9100',
                ['==', ['get', 'type'], 'destination'], '#00e676',
                '#90caf9'
            ],
            'circle-stroke-width': 2,
            'circle-stroke-color': '#ffffff'
        }}
    }});

    // Dynamic Slider Interactivity
    const timeSlider = document.getElementById('time-slider');
    const timeDisplay = document.getElementById('time-display');
    const burnedStat = document.getElementById('burned-stat');
    const closuresStat = document.getElementById('closures-stat');

    const horizons = [
        {{ label: "T = 0 Hours (Ignition)", prop: "prob_0h", closedProp: "closed_0h", burnedPct: "13.3 %", closures: "0 / 34" }},
        {{ label: "T = 3 Hours Forecast", prop: "prob_3h", closedProp: "closed_3h", burnedPct: "33.7 %", closures: "20 / 34" }},
        {{ label: "T = 6 Hours Forecast", prop: "prob_6h", closedProp: "closed_6h", burnedPct: "52.9 %", closures: "24 / 34" }},
        {{ label: "T = 12 Hours Forecast", prop: "prob_12h", closedProp: "closed_12h", burnedPct: "78.4 %", closures: "32 / 34" }}
    ];

    timeSlider.addEventListener('input', (e) => {{
        const h = horizons[e.target.value];
        timeDisplay.innerText = h.label;
        burnedStat.innerText = h.burnedPct;
        closuresStat.innerText = h.closures;

        // Update H3 color & extrusion height
        map.setPaintProperty('h3-extrusion', 'fill-extrusion-color', [
            'interpolate', ['linear'], ['get', h.prop],
            0.0, 'rgba(0,0,0,0)',
            0.2, '#ffeb3b',
            0.5, '#ff9800',
            0.75, '#f44336',
            1.0, '#b71c1c'
        ]);
        map.setPaintProperty('h3-extrusion', 'fill-extrusion-height', ['*', ['get', h.prop], 1800]);

        // Update Road Closures
        map.setFilter('roads-closed', ['==', h.closedProp, true]);
    }});
}});
</script>
</body>
</html>
"""

map_html_path = os.path.join(REPORTS_DIR, "evacuation_map.html")
with open(map_html_path, "w", encoding="utf-8") as f:
    f.write(html_template)

file_size_kb = os.path.getsize(map_html_path) / 1024.0

print("\n=======================================================")
print(f"SUCCESS: Interactive MapLibre GL Dashboard Generated")
print(f"Path: {map_html_path}")
print(f"File Size: {file_size_kb:.2f} KB")
print("=======================================================")


SUCCESS: Interactive MapLibre GL Dashboard Generated
Path: C:\Users\aseem\OneDrive\Dokumen\aerospatium\reports\evacuation_map.html
File Size: 1562.11 KB
